# Session 1 · Part 3 — Spatial features and neighborhoods

**Independent checkpoint:** reload the data, visualize variable features, and save a nearest-neighbor table.


In [ ]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "src" / "dgat_tutorial").is_dir():
        tutorial_root = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter inside the hands-on_tutorial directory.")

sys.path.insert(0, str(tutorial_root / "src"))

from dgat_tutorial.checkpoints import tutorial_paths, write_checkpoint

paths = tutorial_paths(tutorial_root)
print(f"Tutorial root: {paths.root}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

from dgat_tutorial.data import load_tutorial_data
from dgat_tutorial.plotting import plot_spatial_feature

dataset = load_tutorial_data(paths.raw_data, allow_demo=True)
spots, transcripts, proteins = dataset.spots, dataset.transcripts, dataset.proteins

def top_variable_features(matrix, n=3):
    numeric = matrix.select_dtypes(include=[np.number])
    return numeric.var(axis=0).sort_values(ascending=False).head(n).index.tolist()


In [ ]:
features = [
    ("Transcript", transcripts, top_variable_features(transcripts), "magma"),
    ("Protein", proteins, top_variable_features(proteins), "viridis"),
]
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for row, (label, matrix, names, cmap) in enumerate(features):
    for col, feature in enumerate(names):
        plot_spatial_feature(spots, matrix[feature], f"{label}: {feature}", cmap=cmap, ax=axes[row, col])
plt.tight_layout()
feature_figure = paths.figures / "session01_spatial_features.png"
plt.savefig(feature_figure, dpi=160)
plt.show()


In [ ]:
xy = spots[["x", "y"]].to_numpy()
neighbor_count = min(7, len(spots))
distances, neighbor_indices = NearestNeighbors(n_neighbors=neighbor_count).fit(xy).kneighbors(xy)
neighbors = pd.DataFrame({
    "spot_id": spots.index,
    "mean_neighbor_distance": distances[:, 1:].mean(axis=1),
    "nearest_neighbor": spots.index[neighbor_indices[:, 1]],
}).set_index("spot_id")

neighbor_path = paths.results / "session01_spatial_neighbors.csv"
neighbor_figure = paths.figures / "session01_neighborhood_distances.png"
neighbors.to_csv(neighbor_path)
ax = spots.plot.scatter(x="x", y="y", c=neighbors["mean_neighbor_distance"], cmap="cividis", figsize=(5, 4))
ax.set_aspect("equal")
ax.set_title("Mean distance to nearest neighbors")
plt.tight_layout()
plt.savefig(neighbor_figure, dpi=160)
plt.show()

manifest = write_checkpoint(
    "1.3", [neighbor_path, feature_figure, neighbor_figure],
    summary={"spots": len(spots), "neighbors_per_spot": neighbor_count - 1}, start=paths.root
)
print(f"Checkpoint written: {manifest}")


## Checkpoint

Session 1 is complete when the neighborhood table and both spatial figures exist.